# Can you tell where a basalt erupted from its chemistry alone?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2FT7_from_its_chemistry_alone.ipynb).

Basalt is the commonest lava on Earth, and nearly all of it reaches the surface in one of three
places. Under a mid-ocean ridge the mantle rises, the pressure on it falls, and it melts on its
own. Under an island arc the mantle is too cold to melt at all until water squeezed out of a
sinking plate lowers its melting point. Under an ocean island — Hawaii, Iceland — neither applies,
and something hotter appears to be arriving from deeper down. Three plumbing systems, three
magmas; and once the lava has cooled into black rock on a beach, the three look much alike.

Geochemists have claimed since the 1970s that the setting is written in the chemistry, and they
made the claim with three elements on a sheet of graph paper. You have 756 basalts whose
setting somebody established in the field, up to 51 measurements on each, and a
machine that can read all of them at once. The question is not only whether the machine can do it.
It is which chemistry you let it see — and what you give up by letting it see more.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## How this notebook is different

This is a **project track**. It is not a weekly notebook and it does not behave like one.

A weekly notebook shows you a move, walks you through it, and then asks you to make it once
yourself. This one loads the data and reproduces the single result the argument starts from — how
far three elements get you — and then stops helping. From there on every section is a sentence
describing what to find out and an empty cell to find it out in. There is no worked example above
to pattern-match against, because on a real question there never is one.

**There is exactly one self-check in this notebook, and it is on the data loading.** After that,
nothing tells you whether you are right. That is not an oversight and it is not laziness: past the
loading step there is no single right answer here, so a cell that said `assert` would be lying to
you about how research works. What replaces it is the thing researchers actually use — a number
you can predict before you compute it, a result you get twice from different directions, and a
claim you try to break.

**And it does not close.** The last section is a question this course does not know the answer to.
Everything above it is scaffolding; that question is the project.

## What you'll be able to do

**The science.** Say how much of a basalt's tectonic setting is recoverable from its chemistry
alone, and defend a choice of which chemistry to use — against a petrologist who will ask whether
the elements your model leaned on are ones a fifty-million-year-old seafloor rock still remembers.

**The skills.** Compare feature sets honestly: the same model, the same split procedure, and a
sweep over splits rather than one lucky one. Read a model's own account of what it used, and check
that account against how often each column was even measured. Build a stress test for a result
instead of asserting that it is robust.

**The four questions, in order:**

1. How well do three elements tell three settings apart?
2. Which chemistry do you feed the forest?
3. Is the difference you found real, or is it your split?
4. What is the forest actually leaning on?

The open question at the end is not on that list. It is the project; the four above are what you
build to reach it.

## Setup

The table is the compilation from Vermeesch, P. (2006), *Tectonic discrimination of basalts with
classification trees*, Geochimica et Cosmochimica Acta 70, 1839–1848
(doi:10.1016/j.gca.2005.12.016). It ships with the course, so there is nothing to fetch and
nothing to clean: one header row, 51 numeric chemistry columns, and an `affinity`
column holding the setting somebody established in the field.

**One property of this file decides everything you do next, so read it before you go on.** *No
column in it is complete.* The best-measured column is `Sr_ppm` and it is still
6.5% blank; the worst, `Sn_ppm`, is 97.1%
blank. Nobody measures every element on every rock — you measure what your question needed and
what your laboratory could do that year.

So `dropna()` is not an option here, it is a trap: asking for rows complete across all
51 columns leaves **1**. Every accuracy below therefore comes from
filling the holes rather than deleting the rows.

**Imputation:** A blank is not a zero. Fill it with something defensible and say what you filled it with.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

# the Vermeesch (2006) compilation, which ships with the course, so there is nothing to fetch
basalts = pd.read_csv(CACHE + "/trackT7_vermeesch_basalts.csv")

feature_columns = []
for name in basalts.columns:
    if name != "affinity":
        feature_columns.append(name)

print(basalts.shape, "-", len(feature_columns), "chemistry columns and one label")
print(basalts["affinity"].value_counts().to_dict())

In [ ]:
assert basalts.shape == (756, 52), \
    "expected 756 basalts and 52 columns — the file was read wrong"
assert set(basalts["affinity"]) == {"IAB", "MORB", "OIB"}, \
    "the label column should hold exactly three settings"
print(f"✓ the data — {len(basalts)} basalts, {len(feature_columns)} chemistry columns, "
      f"and no column complete (the fullest is {basalts[feature_columns].isna().mean().min():.3f} blank)")

### And that is the last self-check in this notebook

The pipeline is now trustworthy: the file is the file, the labels are the labels. Everything from
here is yours, and nothing will tell you when you have it right.

## How well do three elements tell three settings apart?

The three settings differ in how the melt was made, and that shows up in which elements the melt
carries. Ridge basalt comes from mantle that has already had melt taken out of it once, so it is
poor in the elements that leave easily. Arc basalt is melted by water coming off the sinking
plate, and water carries potassium, rubidium, barium, strontium and lead with it while leaving
niobium, tantalum and titanium behind — so an arc basalt is enriched in the first group and
conspicuously short of the second. Ocean island basalt owes nothing to either mechanism and is the
titanium-rich, niobium-rich one.

Pearce, J.A. and Cann, J.R. (1973), *Tectonic setting of basic volcanic rocks determined using
trace element analyses*, Earth and Planetary Science Letters 19, 290–300, turned that into
diagrams you could draw by hand. They published **two** ternary diagrams: Ti–Zr–Y, and
Ti/100–Zr–Sr/2. Strontium is in the second one, so it was never a forbidden element — it was used.
What its authors said, in the same paper, is that alteration moves strontium around, and that is
why Ti–Zr–Y rather than the strontium diagram is the one people reach for on a basalt that has sat
under the ocean. Shervais, J.W. (1982), *Ti–V plots and the petrogenesis of modern and ophiolitic
lavas*, Earth and Planetary Science Letters 59, 101–118, added a third pair, titanium against
vanadium. Niobium is in none of them; niobium-based discrimination is later work.

Start where they did. Two of Ti–Zr–Y, on a log axis because zirconium spans a factor of
494 across these rocks and a linear axis would pile most of them against the left edge.

In [ ]:
drawn = basalts[['TiO2_wt_percent', 'Zr_ppm', 'Y_ppm', 'affinity']].dropna()

for setting in ["MORB", "OIB", "IAB"]:
    rows = drawn[drawn["affinity"] == setting]
    plt.scatter(rows["Zr_ppm"], rows["TiO2_wt_percent"], s=12, label=setting)

plt.xscale("log")
plt.xlabel("Zr (ppm)")
plt.ylabel("TiO2 (weight percent)")
plt.title(f"{len(drawn)} basalts with Ti, Zr and Y all measured")
plt.legend()
plt.show()

Three fields, and they are real: the arc basalts sit low and to the left, the ocean island basalts
high and to the right. They are also not clean. Ridge and ocean island overlap through the whole
middle of the picture, and any line you draw by hand through that overlap will cut some of both.
Whatever this diagram is worth, it is not worth 100%.

To put a number on it, three moves you have already met. **Baseline:**
Write the dumbest rule you can, first. Any model that cannot beat it is decoration. **Train/test split:** Hide some data from yourself, then check.
**Random forest:** Ask a hundred slightly different trees and take a vote.

The cell below is the whole machine, and it is the only machinery this notebook hands you. Both
helpers take the table first, then the columns, then a `seed` — and `seed` means the same thing in
both places: *the arbitrary choice you did not think about*. In `score` it decides which rocks were
held out; in `importances` it decides which forest grew. Vary it.

In [ ]:
def score(rocks, columns, seed=0):
    """Fit a forest on 70% of `rocks` and report how often it is right on the other 30%."""
    labels = rocks["affinity"]
    X_train, X_test, y_train, y_test = train_test_split(
        rocks[columns], labels, test_size=0.3, random_state=seed, stratify=labels)
    filler = SimpleImputer(strategy="median")
    X_train = filler.fit_transform(X_train)
    X_test = filler.transform(X_test)
    forest = RandomForestClassifier(n_estimators=200, random_state=0)
    forest.fit(X_train, y_train)
    return forest.score(X_test, y_test)


def importances(rocks, columns, seed=0):
    """How much of one forest's decisions each column carried. `seed` picks which forest."""
    filler = SimpleImputer(strategy="median")
    filled = filler.fit_transform(rocks[columns])
    forest = RandomForestClassifier(n_estimators=200, random_state=seed)
    forest.fit(filled, rocks["affinity"])
    return pd.Series(forest.feature_importances_, index=columns).sort_values(ascending=False)


CLASSIC = ['TiO2_wt_percent', 'Zr_ppm', 'Y_ppm']
MAJOR_OXIDES = ['SiO2_wt_percent', 'TiO2_wt_percent', 'Al2O3_wt_percent', 'Fe2O3_wt_percent', 'FeO_wt_percent', 'CaO_wt_percent', 'MgO_wt_percent', 'MnO_wt_percent', 'K2O_wt_percent', 'Na2O_wt_percent']
SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [ ]:
baseline = basalts["affinity"].value_counts().max() / len(basalts)

classic = []
for seed in SEEDS:
    classic.append(score(basalts, CLASSIC, seed))
classic = np.array(classic)

print("always guess the commonest setting:", round(baseline, 3))
print("Ti, Zr and Y over", len(SEEDS), "splits — mean", round(classic.mean(), 3),
      " lowest", round(classic.min(), 3), " highest", round(classic.max(), 3))

So Pearce and Cann's three elements carry most of the way: **80.1%** against a
34.3% baseline, averaged over 10 different splits of the same
756 rocks. Notice what that average is hiding — the ten splits range from
77.1% to 81.9%, a spread of
4.8 points. Any single one of them would have been quotable, and
would have been 4.8 points wrong about its neighbours.

That is the result you are handed, and the last thing this notebook will do for you.

### Predict before you run

You have used 3 of the 51 chemistry columns in the file. The other
48 are sitting there unread — every major oxide, every rare earth, the isotope
ratios.

Change `my_guess` to the number of **accuracy points** you think all 48 of them
would add to 80.1%, and run the cell. You check it in the next section, and a
wrong guess you committed to is worth more than a right answer you were shown.

In [ ]:
my_guess = 5

print("I think the other", len(feature_columns) - 3, "columns are worth", my_guess,
      "accuracy points on top of", round(classic.mean() * 100, 1))

## Which chemistry do you feed the forest?

This is the one real decision in this track, and there is no correct answer to it. Three cuts are
defensible:

- **the classic diagram** — `CLASSIC`, the three elements above, the ones a geologist would have
  plotted by hand and can still interpret;
- **the ten major oxides** — `MAJOR_OXIDES`, the analysis every laboratory runs on every rock as a
  matter of course, so it is the set most likely to exist for a rock you have not seen yet;
- **everything** — `feature_columns`, all 51 of them, including columns that are
  blank for most of the file.

They will not agree. Make the choice, and report what it cost.

### ✏️ Your turn 1

Score all three cuts the way the cell above scored the first one: over every seed in `SEEDS`, not
on one split. Print each cut's mean, lowest and highest accuracy, and draw the three means as a bar
chart with the 34.3% baseline marked on it.

Then answer, in a printed sentence: which of the three would you report to a geochemist, and what
did your guess in *Predict before you run* miss?

In [ ]:
# ← your answer here



Three numbers that disagree, and a fork that has now been taken. Before you defend the choice,
though, there is a prior question, and this course got caught by it on this very file. Two ways of
handling the blanks — deleting the incomplete rows, or filling them — were compared on one split
and differed by 0.105. Swept over ten splits, 0.028 of that was the blanks and 0.077 was the split.

You swept ten seeds above, so you have what you need to check your own.

## Is the difference you found real, or is it your split?

An accuracy is a property of a model *and* of the 227 rocks that happened to land in the
held-out half. Two feature sets compared on one split are two numbers with an unknown amount of coin-flip
in them. Compared on the same ten splits, the coin-flip is largely shared, and the difference is
the thing you can actually talk about.

### ✏️ Your turn 2

Take the two cuts furthest apart — the classic diagram and everything — and this time keep the
**per-seed difference**, not the two means: one number for each seed, `everything minus classic`.

Print the mean difference, its smallest and largest value across the seeds, and how many of the
10 seeds gave a difference of zero or less. Draw the ten differences however makes the
spread visible.

Then answer in a printed sentence: is the gap you reported in *Your turn 1* bigger than the wobble
between splits, and how much of the number you would quote is the split rather than the chemistry?

In [ ]:
# ← your answer here



Whatever your sweep said about that comparison, it was a comparison between three columns and
48 more. The place a sweep decides an answer rather than refining it is where
the two things being compared are close together — and there is a comparison like that waiting,
because those 48 columns did not contribute equally.

### ✏️ Your turn 3

Go through the 48 chemistry columns that are **not** in `CLASSIC`, one at a time.
For each, score the classic three *plus that one column*, swept over `SEEDS`, and keep the mean.

Print the five that help most, with their accuracies, and print how much of the whole
48-column gain from *Your turn 2* the single best one recovers on its own.

That is 48 columns times ten splits, so the cell has real work to do; a loop
inside a loop is the plainest way to write it, and it does not have to be quick.

Then answer in a printed sentence: is the gain you found spread across the file, or does it live in
a small number of columns — and does the best single column have a clear lead over the runners-up,
or is it a photo finish?

In [ ]:
# ← your answer here



## What is the forest actually leaning on?

You have just asked which column *helps most when added to three others*. A forest trained on all
51 at once will answer a related but different question — which columns it split on,
and how often — and there is no guarantee the two agree.

**Feature importance:** How much of a forest's decisions each column carried — the model's own account of what it used.

Two warnings before you read one. A forest's importance ranking wobbles between forests the way an
accuracy wobbles between splits, so one forest's top ten is not a result. And a column that is
blank on most of the file cannot be important however informative it is, so the ranking has to be
read against how often each column was measured at all.

### ✏️ Your turn 4

Run `importances(basalts, feature_columns, seed)` for several forest seeds and average the
51 numbers across them. Draw the top ten as a horizontal bar chart, and print each
one beside the fraction of the file where that column is blank — `basalts[name].isna().mean()`.

Then answer in a printed sentence, using a criterion instead of an impression. The elements
seawater and low-grade metamorphism move around long after the lava has cooled are the ones with
large ions and a single charge — **strontium, potassium, rubidium, barium, caesium**. Titanium,
zirconium, yttrium and niobium stay put. Where do the mobile elements land in *your* ranking, and
what would that mean for a model shown a rock that had spent fifty million years on the seafloor?

In [ ]:
# ← your answer here



If a mobile element came out near the top of your ranking, you have met the tension this track
exists for, and it is older than machine learning. Pearce and Cann had strontium in one of their two diagrams and warned about it in the same paper; the community
kept the Ti–Zr–Y diagram and largely dropped the strontium one, not because strontium says less but
because what it says stops being trustworthy once seawater has been through the rock.

A forest cannot make that distinction. It cannot tell an altered sample from a fresh one, so it
takes every measurement at face value and is rewarded for doing so on a compilation of mostly fresh
rocks. Choosing the classic diagram over the full chemistry is choosing to give up accuracy you can
measure in exchange for robustness you cannot — at least, not yet, and not from anything you have
computed so far.

### ✏️ Your turn 5

Two or three paragraphs, quoting **your own numbers** — the three accuracies from *Your turn 1*,
the per-seed range from *Your turn 2*, and where the mobile elements sat in *Your turn 4*.

1. Which cut would you report, and to whom? Say what a reader loses by taking it, and name the kind
   of rock on which you would expect your reported accuracy to be wrong.
2. Pearce and Cann's community preferred the diagram that scores worse. On the evidence you have
   produced, is that preference defensible — and what have you actually measured about it, as
   against what you have assumed?

*(Double-click this cell and replace this line with your answer.)*

## The question, answered

Yes, largely. 34.3% is what guessing gets you; three elements chosen in 1973 get
80.1%; the ten oxides every laboratory measures get 90.6%;
and all 51 columns, holes filled rather than rows deleted, get
95.3%. The setting really is written in the chemistry.

What the numbers do not settle is which of those you should report, because the best-scoring model
leans hardest on the measurements a fifty-million-year-old rock is least likely to have kept.

## What track T7 leans on

**The question.** Can you tell where a basalt erupted from its chemistry alone?

Nothing here is new. These are the weeks to look back at while you work, and the wording is the course's own.

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Baseline** | Write the dumbest rule you can, first. Any model that cannot beat it is decoration. |
| **Train/test split** | Hide some data from yourself, then check. |
| **Random forest** | Ask a hundred slightly different trees and take a vote. |
| **Imputation** | A blank is not a zero. Fill it with something defensible and say what you filled it with. |
| **Feature importance** | How much of a forest's decisions each column carried — the model's own account of what it used. |
| **Missingness as data** | Which measurements are missing is itself a record of who measured the sample, and a model will use it if you let it. |

### Code you will reach back for

| Function | What it does |
|---|---|
| `train_test_split(X, y, test_size=0.3, random_state=n, stratify=y)` | cut the rows into a training set and a held-out set, keeping the class balance |
| `SimpleImputer(strategy="median")` | put the middle value of a column into that column's holes |
| `filler.fit_transform(X_train) / filler.transform(X_test)` | learn the fill values on the training set only, then apply the same ones to the test set |
| `RandomForestClassifier(n_estimators=200, random_state=0)` | a hundred slightly different trees, voting |
| `forest.feature_importances_` | one number per column: how much of the forest's decisions it carried |
| `pd.Series(values, index=names)` | a column of numbers with a name on every row |
| `plt.bar(x, heights) / plt.barh(labels, values)` | one bar per item; barh when the labels are words |
| `column.value_counts()` | how often each value appears |
| `column.isna()` | a mask marking where the file had nothing |
| `table.sort_values(by)` | put the rows in order by one column |
| `np.random.default_rng(seed)` | a random-number generator you can reproduce — the same seed gives the same draws |

## What your project must contain

Five sections, empty below, required of **every** EPS 88 project regardless of track. They are
headed here so the shape of a good answer is visible while you work. Fill them in as you go; they
are not a write-up you do at the end.

### ✏️ 1 · A one-sentence answer

Your claim and its uncertainty, in one sentence, at the top of your report. If you cannot put a
number and a range in it, you do not have a result yet. On this track the range is not optional:
every accuracy you have is a mean over splits with a spread beside it.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 2 · The trivial baseline

Before any statistic, state the dumbest answer to your question and what it gives. Every later
number is reported against it.

On this track you computed it in the first minute — guess the commonest of the three settings — and
the three classes are near enough equal that it is a real floor rather than a formality. Quote
every accuracy in your report against it, and say what each step of extra chemistry bought over it.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 3 · Split by structure

Earth data are correlated in space and in time, so whatever you split, resample or count as
independent has to be split along the structure that is really there — never at random across rows.

This track splits at random, and you should say why that is a problem here rather than repeating
that it is one. These 756 rows are analyses from many separate published studies; several rows can
be the same lava flow, the same island, the same cruise. A random split puts two analyses of one
rock on opposite sides of it. The file carries no study column, so you cannot fix this by splitting
on one. Say what you would need in order to, and what your accuracy means until you have it.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 4 · What I got wrong

What failed, and what you believed before it failed. Honest failure is graded; a faked success is
not. Your *Predict before you run* guess belongs here if it was wrong.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 5 · AI disclosure

Which tool, what you asked it, what you changed in what it gave you, and how you checked that the
result was true.

*(Double-click this cell and replace this line with your answer.)*

## The open question

> **Is that the right trade for a rock that has sat on the seafloor for 50 million years, and can you MEASURE the robustness it buys rather than assuming it?**

Nobody grading this knows the answer, and neither does the literature settle it. Everything above
is the scaffolding; this is the project.

Here is exactly what is established and what is not. Established: on this compilation, more
chemistry classifies better, by 15.2 points over the classic diagram, on every one
of ten splits. Established: the model's second-most-used column is one that seawater moves. **Not
established: anything at all about the robustness the classic diagram is supposed to buy.** That
half of the trade has been argued from mechanism in every sentence above, including the ones in
this notebook, and never measured.

Four directions, none of them worked out here:

1. **Score the diagram, not the elements.** Pearce and Cann plot a *ternary*: Ti/100, Zr and 3×Y,
   each divided by their sum. Only the ratios survive that; the overall abundance is thrown away.
   Feeding a forest raw concentrations gives it something the published diagram never had. Convert
   the three columns to ternary fractions, re-score, and find out how much of the
   80.1% was the diagram and how much was the extra information.
2. **Measure the robustness instead of assuming it.** You cannot get an altered basalt out of this
   file, but you can make one: multiply the mobile columns by a random factor and see which cut
   degrades. That is the direction *Your turn 6* takes, and it settles less than it looks like it
   does, because the size of the factor is a free parameter and this file cannot tell you the right
   one. The harder version, which nobody here has run, trains on fresh rocks and tests on altered
   ones — the real deployment case, and a strictly nastier test than altering both halves.
3. **Find out how independent 756 rows are.** Which elements a row has is a fingerprint
   of which study measured it, and studies tend to be about one setting at a time. Cluster the rows
   by their pattern of blanks, check how the settings fall across the clusters, and split by
   cluster instead of at random. If the accuracy drops, part of what you measured was bookkeeping.
4. **Ask what would settle it.** The measurement that would decide the whole question is one this
   file does not carry: an alteration index — loss on ignition, or a measured degree of
   sea-floor weathering — on every sample. What would you do with such a column if you had it, and
   how many altered samples would you need before you could say which cut to trust?

### ✏️ Your turn 6 — the first move

Before you close this notebook: in a few sentences, name the **one** measurement you would make
first. What would it show if the classic diagram's robustness is worth its
15.2-point cost, what would it show if it is not, and what number would change your
mind?

Then make the measurement, in the cell below the prose.

*(Double-click this cell and replace this line with your answer.)*

In [ ]:
# ← your answer here

